In [ ]:
from pathlib import Path

from paddleocr import PPStructureV3


PDF_PATH = Path(
    r"C:\Programming\multi-agent-system\data\uploads\pdf\firma\rechnung_firma_test.pdf"
)

OUTPUT_DIR = Path("output")


def extract_pdf_structure(
    pdf_path: str | Path,
    output_dir: str | Path,
) -> None:
    """
    Extrahiert strukturierte Inhalte und Tabellen aus einer PDF
    mit PaddleOCR PPStructureV3.
    """

    # --------------------------------------------------
    # Pfade normalisieren
    # --------------------------------------------------

    pdf_path = Path(pdf_path)
    output_dir = Path(output_dir)

    if not pdf_path.exists():
        raise FileNotFoundError(
            f"PDF nicht gefunden: {pdf_path.resolve()}"
        )

    if not pdf_path.is_file():
        raise ValueError(
            f"Der angegebene Pfad ist keine Datei: {pdf_path}"
        )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    # --------------------------------------------------
    # PaddleOCR Pipeline
    # --------------------------------------------------

    pipeline = PPStructureV3(
        lang="de",
        device="cpu",

        # Nicht benötigte Verarbeitung deaktivieren
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_textline_orientation=False,

        use_formula_recognition=False,
        use_seal_recognition=False,
        use_chart_recognition=False,

        # Tabellenanalyse aktivieren
        use_table_recognition=True,

        # Verhindert Probleme mit oneDNN / MKLDNN
        enable_mkldnn=False,
    )

    # --------------------------------------------------
    # PDF verarbeiten
    # --------------------------------------------------

    results = pipeline.predict(
        input=str(pdf_path),
    )

    # --------------------------------------------------
    # Ergebnisse je Seite speichern
    # --------------------------------------------------

    page_count = 0

    for page_number, result in enumerate(
        results,
        start=1,
    ):
        page_count += 1

        print("=" * 80)
        print(f"SEITE {page_number}")
        print("=" * 80)

        result.print()

        page_dir = output_dir / f"page_{page_number}"

        page_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        # JSON-Struktur
        result.save_to_json(
            save_path=str(page_dir)
        )

        # Tabellen als Excel
        result.save_to_xlsx(
            save_path=str(page_dir)
        )

        # Tabellen / Struktur als HTML
        result.save_to_html(
            save_path=str(page_dir)
        )

    print(
        f"✅ {page_count} Seite(n) verarbeitet."
    )
    print(
        f"📁 Ergebnisse gespeichert unter: "
        f"{output_dir.resolve()}"
    )


if __name__ == "__main__":

    try:
        extract_pdf_structure(
            pdf_path=PDF_PATH,
            output_dir=OUTPUT_DIR,
        )

    except FileNotFoundError as error:
        print(f"❌ Datei nicht gefunden: {error}")

    except Exception as error:
        print(
            f"❌ Fehler bei der Verarbeitung: "
            f"{type(error).__name__}: {error}"
        )